# Integración final de Datasets
En este cuaderno se unen los 3 datasets preprocesados originarios de fuentes gubernamentales nacionales e internacionales para la creación del dataset final.

El dataset proveerá los registros necesarios que expliquen el contexto metereológico que provocó la cantidad de casos de dengue en cada semana del periodo histórico 2000-2025

## Importación de librerías

In [1]:
from pathlib import Path
import pandas as pd

## Importar datasets originales

In [2]:
# Localizamos Datasets desde el directorio de trabajo o sus carpetas superiores
ruta_datasets = next(
    (ruta / 'Datasets' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'Datasets').is_dir()),
    None
)
if ruta_datasets is None:
    raise FileNotFoundError("No se encontró la carpeta 'Datasets' del proyecto.")

# Importamos los tres datasets semanales preprocesados
df_dengue_semanal = pd.read_csv(ruta_datasets / 'casos_dengue_piura_2000_2025_semanal.csv')
df_clima_semanal = pd.read_csv(ruta_datasets / 'clima_piura_2000_2025_semanal.csv')
df_enfen_semanal = pd.read_csv(ruta_datasets / 'temperatura_superficial_mar_piura_2000_2025_semanal.csv')


## Preparación del archivo maestro

In [3]:
# 1. PREPARACIÓN DEL DATASET MINSA COMPLETO (2000 - 2025)
# El archivo MINSA ya contiene los casos agrupados por año y semana.
serie_dengue_total = df_dengue_semanal.copy()

# 2. CREACIÓN DEL CALENDARIO MAESTRO (Espina dorsal temporal 2000-2025)
anos_rango = list(range(2000, 2026))
semanas_rango = list(range(1, 53)) # Estándar de 52 semanas continuas por año (1,352 semanas en total)

calendario_maestro = pd.MultiIndex.from_product(
    [anos_rango, semanas_rango], 
    names=['ano', 'semana']
).to_frame().reset_index(drop=True)

# Aseguramos tipos enteros en todas las llaves antes del cruce
for dataset in (serie_dengue_total, df_clima_semanal, df_enfen_semanal):
    dataset['ano'] = dataset['ano'].astype(int)
    dataset['semana'] = dataset['semana'].astype(int)

## Construcción del Dataset final con merge
Se usa merge para unir cada uno de los datasets usando los campos 'ano' y 'semana' como identificadores.
Posteriormente se realiza una imputación de datos para aquellos registros semanales donde no hubo reportes o no hubo lecturas

In [4]:
# 3. TRIPLE MERGE (Ensamblaje del DataMart)
df_final = calendario_maestro.merge(serie_dengue_total, on=['ano', 'semana'], how='left')
df_final = df_final.merge(df_clima_semanal, on=['ano', 'semana'], how='left')
df_final = df_final.merge(df_enfen_semanal, on=['ano', 'semana'], how='left')

# 4. REGLAS DE IMPUTACIÓN DE NEGOCIO Y FÍSICA
# Regla A: Si el MINSA no reportó filas en una semana, hubo 0 casos confirmados
df_final['Casos_Confirmados'] = df_final['Casos_Confirmados'].fillna(0).astype(int)

# Regla B: Si alguna semana puntual no tuvo lectura satelital/climática, aplicamos forward/backward fill
columnas_ambientales = ['Precipitacion', 'Temp_Max', 'Temp_Min', 'TSM_Nino12', 'Anomalia_Nino12']
df_final[columnas_ambientales] = df_final[columnas_ambientales].ffill().bfill()

In [6]:
# 5. VERIFICACIÓN DE CALIDAD DEL DATAMART
print("--- AUDITORÍA DEL DATASET INTEGRADO (2000-2025) ---")
print(f"Dimensiones finales (Semanas x Columnas): {df_final.shape}")
print("\nConteo de valores nulos por columna:")
print(df_final.isna().sum())

print("\n--- MUESTRA EN PLENO PICO DEL NIÑO COSTERO (2017) ---")
print(df_final[(df_final['ano'] == 2017) & (df_final['semana'].between(10, 16))])

--- AUDITORÍA DEL DATASET INTEGRADO (2000-2025) ---
Dimensiones finales (Semanas x Columnas): (1352, 8)

Conteo de valores nulos por columna:
ano                  0
semana               0
Casos_Confirmados    0
Precipitacion        0
Temp_Max             0
Temp_Min             0
TSM_Nino12           0
Anomalia_Nino12      0
dtype: int64

--- MUESTRA EN PLENO PICO DEL NIÑO COSTERO (2017) ---
      ano  semana  Casos_Confirmados  Precipitacion  Temp_Max  Temp_Min  \
893  2017      10                591          216.6     30.80     23.26   
894  2017      11                775          109.7     30.51     22.99   
895  2017      12               1061          239.7     30.60     23.41   
896  2017      13               1607           78.1     30.57     22.89   
897  2017      14               2537           40.0     30.76     22.19   
898  2017      15               3375            0.4     31.93     22.51   
899  2017      16               4329           35.7     30.46     22.26   

     

# Exportación del Dataset Final 

In [8]:
# Guardar archivo CSV final en la carpeta data del proyecto
from pathlib import Path
ruta_datasets = next(
    (ruta / 'data' for ruta in (Path.cwd(), *Path.cwd().parents)
     if (ruta / 'data').is_dir()),
    None
)
if ruta_datasets is None:
    raise FileNotFoundError("No se encontró la carpeta 'data' del proyecto.")

ruta_csv = ruta_datasets / 'df_final.csv'
df_final.to_csv(ruta_csv, index=False)
print(f"Dataset guardado en: {ruta_csv}")

Dataset guardado en: c:\Users\fabri\OneDrive\Desktop\PROYECTO-IA\data\df_final.csv
